## convert docx into txt


In [ ]:
import os
from docx import Document

# Function to convert .docx to .txt
def docx_to_mmd(docx_path, txt_path):
    # Open the .docx file
    doc = Document(docx_path)
    
    # Extract text from all paragraphs in the document
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    
    # Save the text to a .txt file
    with open(txt_path, "w") as txt_file:
        txt_file.write(text)
    print(f"Converted {docx_path} to {txt_path}")

# Folder paths
docx_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Math_s Word File"
txt_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/ocr_human/math"

# Ensure the output folder exists
os.makedirs(txt_folder, exist_ok=True)

# Loop through all files in the input folder
for filename in os.listdir(docx_folder):
    if filename.endswith(".docx"):  # Check if it's a .docx file
        # Define the full path for the .docx and .txt files
        docx_path = os.path.join(docx_folder, filename)
        txt_filename = os.path.splitext(filename)[0] + ".txt"  # Convert .docx to .txt
        txt_path = os.path.join(txt_folder, txt_filename)
        
        # Convert the file
        docx_to_txt(docx_path, txt_path)


Converted /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Math_s Word File/10_100210404319737581171703337845.docx to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/ocr_human/math/10_100210404319737581171703337845.txt
Converted /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Math_s Word File/11_10021019941080491171694789012.docx to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/ocr_human/math/11_10021019941080491171694789012.txt
Converted /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Math_s Word File/14_10021393251035351171693723024 (Mohit Sir).docx t

## to copy pdf and put it/filter it

In [19]:
import os
import shutil

# Define the directories
doc_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Physics Word File"
pdf_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf/physics"
destination_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy"

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Get all .doc files in the doc_folder
doc_files = [f for f in os.listdir(doc_folder) if f.endswith('.docx')]

# Get all .pdf files in the pdf_folder
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

# Compare filenames (excluding the extension) and copy matching .pdf files
for doc_file in doc_files:
    # Remove the '.doc' extension to get the base filename
    base_name = os.path.splitext(doc_file)[0]

    # Find the matching .pdf file
    matching_pdf = f"{base_name}.pdf"
    
    if matching_pdf in pdf_files:
        # Get the full path of the .pdf file to be copied
        pdf_file_path = os.path.join(pdf_folder, matching_pdf)

        # Copy the matching PDF to the destination folder
        shutil.copy(pdf_file_path, os.path.join(destination_folder, matching_pdf))
        print(f"✅ Copied {matching_pdf} to {destination_folder}")
    else:
        print(f"⚠️ No matching PDF found for {doc_file}")

print("📂 All matching PDFs have been copied.")


✅ Copied 10_10021138351083421111694954514.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy
✅ Copied 04_100210408219741901111703509552.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy
✅ Copied 15_10021105101083421111694960631.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy
✅ Copied 13_10021000191039611111693742763.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy
✅ Copied 09_1002114885961841111690700733.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf_chosen/phy
✅ Copied 06_100210285819741901111703513676.pdf to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjectiv

## to send pdf to gemini get ocr output

In [ ]:
import os
import fitz  # PyMuPDF for PDF processing
from PIL import Image
import json
import google.generativeai as genai
from dotenv import load_dotenv
import base64
import time
import numpy as np
import cv2

# === Load API Key ===
load_dotenv()
model_name = "gemini-2.5-pro"
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))
model = genai.GenerativeModel(model_name)

# === Prompt for Gemini ===
PROMPT = """
### System Instruction

**Role**: You are a meticulous digital archivist tasked with transcribing handwritten student answer sheets.

**Core Task**: Your goal is to create a perfect digital copy of the student's work.
**Important Rule**: **DO NOT** correct any spelling, punctuation, or grammatical errors. This includes cases where words might seem misspelled, such as "metabolities" instead of "metabolites". **Preserve all text exactly as it appears in the PDF**, even if there are apparent mistakes or inconsistencies.

---

### IGNORE ALL STRIKETHROUGH TEXT
Any portion of text that has a line through it (strikethrough) MUST BE COMPLETELY REMOVED OR IGNORED from the output. Do not include it.
---

### Output Format
- The output MUST be a single, valid JSON array containing one object per main question.
- Do NOT include any text or explanations outside of the JSON array.

**Example of a valid JSON object:**
```json
[
  {
    "question_number": 1,
    "question_text": "This is the answer to the first part which contains a PV curve <diagram_1>.",
    "diagrams": [
      {
        "id": "diagram_1",
        "question_image_description": " descripiton of the image ",
        "coordinates": "0.5,0.5,0.2,0.3",
        "diagram_class": "graph or diagram",
        "page_number": 3
      }
    ],
    "pages": [2, 3]
  }
]

```
**Schema Definitions:**
- `question_number` (integer): The main question number. question_number should be in increasing order which is the question number of the content,Question numbers may appear in various formats—such as compound forms like (1),  (2), or simple forms like 7, 8, 9. Always preserve the original numbering exactly as it appears in the document..
- `question_text` (string):ocr content of the question, including <diagram_1> if any diagram exists.
- `diagrams` (array): A list of diagram objects. Leave as an empty array `[]` if none.
  - `id` (string): The diagram identifier from the text.
  - `coordinates` (string): "x_mid,y_mid,width,height", with values normalized between 0 and 1 relative to image dimensions.
  - `diagram_class` (string): The class of the diagram.
  -- `question_image_description` (string): The description of the question image.
  - `page_number` (integer): The page where the diagram is located.
- `pages` (array): A list of all page numbers on which any part of the question appears.

- Understand the context to group full question-answer blocks together.
- Maintain the output structure and avoid inserting extra commentary or descriptions.
- follow the question ordering 
- avoid any side calculations just copy the main answer .
-- Ignore any template, header, footer,Dates on the page  or decorative elements such as 'Date', 'Page', and similar non-content areas that do not contribute to the main educational material
-Strictly Ignore the strikethrough words from the output 

"""


# === Set the dimension value ===
dim = 768  # Define the dimension value

# === Resize Image Function ===
def resize_image(image, dim=dim, save_path=None):
    image1 = np.array(image.convert('RGB'))  # Ensure the image is in RGB mode
    original_size = image1.shape  # (height, width, channels)
    image1 = image1.mean(axis=2)  # Convert image to grayscale
    h, w = image1.shape
    if w > h:
        new_w = dim
        new_h = int(h * (dim / w))
    else:
        new_h = dim
        new_w = int(w * (dim / h))
    resized_image = cv2.resize(image1, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_image_pil = Image.fromarray(resized_image)
    resized_image_pil = resized_image_pil.convert('RGB')  # Convert to RGB before saving
    if save_path:
        resized_image_pil.save(save_path)
    return original_size, (new_h, new_w), resized_image_pil

# === Load PDF, Convert Pages to Images ===
def pdf_to_images(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    images = []
    num_pages = doc.page_count
    for i in range(num_pages):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=300)
        img_path_default = os.path.join(output_folder, f"page_{i + 1}.jpeg")
        pix.save(img_path_default)
        images.append(img_path_default)
    return images, num_pages

# === Load Base64 Images ===
def load_base64_images(folder_path, num_pages):
    b64_list = []
    for i in range(num_pages):
        img_filename_dim = f"DIM_{dim}_PAGE_{i + 1}.jpeg"  # Use dim variable here
        img_filename_default = f"page_{i + 1}.jpeg"  # Old naming convention

        img_path_dim = os.path.join(folder_path, img_filename_dim)
        img_path_default = os.path.join(folder_path, img_filename_default)

        if os.path.exists(img_path_dim):
            path = img_path_dim
        elif os.path.exists(img_path_default):
            path = img_path_default
        else:
            print(f"❌ Image {img_filename_dim} or {img_filename_default} not found in {folder_path}")
            continue

        with open(path, "rb") as f:
            b64_list.append(base64.b64encode(f.read()).decode())
    return b64_list

# === Batch send to Gemini ===
def send_to_gemini(resized_images_objs):
    try:
        response = model.generate_content([PROMPT] + resized_images_objs)  # Batch processing
        raw = response.text.strip()
        cleaned = raw.strip('```json').strip('```').strip()
        parsed = json.loads(cleaned)
        return parsed
    except Exception as e:
        print(f"❌ Failed to process images: {e}")
        return None

# === Main Process ===
def main(pdf_file_path):
    # Extract folder name from the PDF file name
    pdf_filename = os.path.splitext(os.path.basename(pdf_file_path))[0]
    
    # Define the output folder path using the PDF filename
    output_folder = os.path.join("/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/gemini_2.5_pro_768/Biology", pdf_filename)  # Adjust path as needed
    os.makedirs(output_folder, exist_ok=True)

    # Step 1: Convert PDF to images
    images, num_pages = pdf_to_images(pdf_file_path, output_folder)

    # Step 2: Resize images
    resized_images = []
    for page_num in range(len(images)):
        img_path = images[page_num]
        original_size, new_size, resized_image = resize_image(Image.open(img_path), dim=dim)
        
        # Save resized image
        img_filename_dim = f"DIM_{dim}_PAGE_{page_num + 1}.jpeg"
        resized_image.save(os.path.join(output_folder, img_filename_dim))
        
        resized_images.append(resized_image)

    # Step 3: Load base64 images
    images_b64 = load_base64_images(output_folder, len(images))

    # Step 4: Send to Gemini for OCR
    results = send_to_gemini(resized_images)

    if results:
        output_json_filename = f"{pdf_filename}.json"
        json_path = os.path.join(output_folder, output_json_filename)

        # Save the OCR results
        with open(json_path, "w") as f:
            json.dump(results, f, indent=3)

        print(f"OCR results saved to {json_path}")
    else:
        print("❌ No results from Gemini OCR")

# Running the process with a given PDF file path
pdf_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pdf/pdf_GD/bio/08_1002103647961841141690697632.pdf"  # Replace with your PDF file path
main(pdf_file_path)


OCR results saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/gemini_2.5_pro_768/Biology/08_1002103647961841141690697632/08_1002103647961841141690697632.json


## TO GET THE MMD OF COMPARISON BETWEEN GEMINI AND GROUND

In [5]:
!pip install --upgrade google-generativeai


In [6]:
import os
import json
from dotenv import load_dotenv
import google.generativeai as genai
from google.generativeai import Part  # ✅ Correct import

# === Load API Key ===
load_dotenv()

model_name = "gemini-2.5-pro"

# Configure genai
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Instantiate model
model = genai.GenerativeModel(model_name)


def send_files_and_prompt(txt_file_path, json_file_path, pdf_path, prompt):
    # Read the text file
    with open(txt_file_path, 'r', encoding='utf-8') as file:
        txt_content = file.read()

    # Read the JSON file
    with open(json_file_path, 'r', encoding='utf-8') as file:
        json_content = json.load(file)

    # Read the PDF file as binary
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    pdf_part = Part.from_data(data=pdf_bytes, mime_type="application/pdf")

    # Compose a single prompt string
    full_prompt = (
        f"{prompt}\n\n"
        f"Text content:\n{txt_content}\n\n"
        f"JSON content:\n{json.dumps(json_content, indent=2)}"
    )

    # Send the data to the model for processing
    try:
        response = model.generate_content([full_prompt, pdf_part])
        generated_text = response.text

        # Save the generated output to a text file
        output_txt_file_path = "YESSSSoutput_generated.txt"
        with open(output_txt_file_path, 'w', encoding='utf-8') as output_file:
            output_file.write(generated_text)
        
        print(f"Output saved to {output_txt_file_path}")
    except Exception as e:
        print(f"Error in generating response: {str(e)}")

# Example usage
txt_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/Converting Handwriting PDF to Word File-2/Biology Word file/01_1002115268961841141690701450.txt"
json_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/Gemini_ocr_output_768_v1_p3/B01_1002115268961841141690701450-gemini-2.5-pro/B01_1002115268961841141690701450-gemini-2.5-pro_output.json"
pdf_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/subject_wise-pdf/B01_1002115268961841141690701450.pdf"
prompt = """
Role: You are a highly accurate Quality Assurance (QA) Engine for OCR systems.

Context: You will be provided with three documents for each task:

A Ground Truth PDF: This is the master document, the single source of truth. Its contents are for your internal reference only and must not be displayed in the final output.
A JSON File: Contains text extracted from the PDF by an LLM.
A TXT File: Contains text extracted from the PDF by a human.
Objective: Your sole purpose is to compare the JSON and TXT files against the hidden Ground Truth PDF. For each corresponding solution, you must determine which file—JSON or TXT—perfectly matches the ground truth.

Specific Instructions:

Perform an Internal Three-Way Comparison: For each solution, you will silently compare the text from the JSON file and the TXT file against the text from the Ground Truth PDF.
From the JSON file, use the value of the "ocr_text" key.
From the TXT file, use the text between the <sol_start ...> and <sol_end> tags.
Report Your Verdict: Based on your internal comparison, you will populate a table that declares a verdict and provides a justification. Do not show the ground truth text in your response.
Required Output Format:

Present your verdict in a structured table. The table must have the following five columns:

1. Question Number: The solution number being compared.
2. JSON Version (LLM OCR): The raw text as it appears in the JSON ocr_text.
3. TXT Version (Human OCR): The raw text as it appears in the TXT solution block.
4. File Matching Ground Truth: State which file is correct. Use one of these specific labels: JSON, TXT, Both, or Neither.
5. Justification: Provide a brief explanation for your verdict. If one file is correct, state that it matches the ground truth and briefly describe the error(s) in the other file. If neither is correct, describe the errors in both.
"""

send_files_and_prompt(txt_file_path, json_file_path, pdf_path, prompt)

ImportError: cannot import name 'Part' from 'google.generativeai' (/opt/anaconda3/envs/Layout_env/lib/python3.9/site-packages/google/generativeai/__init__.py)

In [11]:
import os
import json
from dotenv import load_dotenv
import google.generativeai as genai

# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model = genai.GenerativeModel("gemini-2.5-pro")  # safer choice than "2.5-pro" for now

def send_files_and_prompt(txt_file_path, json_file_path, pdf_path, prompt):
    # Read text
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        txt_content = f.read()

    # Read JSON
    with open(json_file_path, 'r', encoding='utf-8') as f:
        json_content = json.load(f)

    # Upload the PDF
    pdf_upload = genai.upload_file(pdf_path)
    print(f"✅ Uploaded PDF: {pdf_upload.uri}")

    # Combine all content into a prompt
    full_prompt = (
        f"{prompt}\n\n"
        f"<Human OCR>\n{txt_content}\n\n"
        f"<LLM OCR>\n{json.dumps(json_content, indent=2)}"
    )

    try:
        response = model.generate_content(
            [full_prompt, pdf_upload],
            generation_config={"temperature": 0.2},
        )
        generated_text = response.text

        # Save output
        with open("YESSSSoutput_generated.txt", 'w', encoding='utf-8') as out_file:
            out_file.write(generated_text)

        print("🎉 Output saved to YESSSSoutput_generated.txt")

    except Exception as e:
        print(f"❌ Error generating response: {e}")

# === Your Inputs ===
txt_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/Converting Handwriting PDF to Word File-2/Biology Word file/01_1002115268961841141690701450.txt"
json_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/Gemini_ocr_output_768_v1_p3/B01_1002115268961841141690701450-gemini-2.5-pro/B01_1002115268961841141690701450-gemini-2.5-pro_output.json"
pdf_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/subject_wise-pdf/B01_1002115268961841141690701450.pdf"

prompt = """
Role: You are a highly accurate Quality Assurance (QA) Engine for OCR systems.

Context: You will be provided with three documents for each task:

A Ground Truth PDF: This is the master document, the single source of truth. Its contents are for your internal reference only and must not be displayed in the final output.
A JSON File: Contains text extracted from the PDF by an LLM.
A TXT File: Contains text extracted from the PDF by a human.
Objective: Your sole purpose is to compare the JSON and TXT files against the hidden Ground Truth PDF. For each corresponding solution, you must determine which file—JSON or TXT—perfectly matches the ground truth-PDF.

Specific Instructions:
Your most important rule: The Ground Truth PDF is the absolute, infallible standard; you must not correct any errors it may contain, but instead use it as the perfect reference to judge which file is its most exact replica.
Perform an Internal Three-Way Comparison: For each solution, you will silently compare the text from the JSON file and the TXT file against the text from the Ground Truth PDF.
From the JSON file, use the value of the "ocr_text" key.
From the TXT file, use the text between the <sol_start ...> and <sol_end> tags.
Report Your Verdict: Based on your internal comparison, you will populate a table that declares a verdict and provides a justification. Do not show the ground truth text in your response.
Required Output Format:

Present your verdict in a structured table. The table must have the following five columns:

1. Question Number: The solution number being compared.
2. JSON Version (LLM OCR): The raw text as it appears in the JSON ocr_text.
3. TXT Version (Human OCR): The raw text as it appears in the TXT solution block.
4. File Matching Ground Truth: State which file is correct. Use one of these specific labels: JSON, TXT, Both, or Neither.
5. Justification: Provide a brief explanation for your verdict. If one file is correct, state that it matches the ground truth and briefly describe the error(s) in the other file. If neither is correct, describe the errors in both.
"""
send_files_and_prompt(txt_file_path, json_file_path, pdf_path, prompt)


✅ Uploaded PDF: https://generativelanguage.googleapis.com/v1beta/files/zucqegk0t9o9
🎉 Output saved to YESSSSoutput_generated.txt


In [9]:
import os
import json
from dotenv import load_dotenv
import google.generativeai as genai

# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model = genai.GenerativeModel("gemini-2.5-pro")

def send_files_and_prompt(txt_file_path, json_file_path, prompt):
    # Read TXT file (Human OCR)
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        txt_content = f.read()

    # Read JSON file (LLM OCR)
    with open(json_file_path, 'r', encoding='utf-8') as f:
        json_content = json.load(f)

    # Compose prompt content
    full_prompt = (
        f"{prompt}\n\n"
        f"<Human OCR>\n{txt_content}\n\n"
        f"<LLM OCR>\n{json.dumps(json_content, indent=2)}"
    )

    try:
        response = model.generate_content(
            full_prompt,
            generation_config={"temperature": 0.2},
        )
        generated_text = response.text

        # === Construct output filename ===
        base_name = os.path.splitext(os.path.basename(txt_file_path))[0]
        
        # Set the specific output folder for saving the file
        output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation"
        
        # Ensure the output folder exists
        os.makedirs(output_folder, exist_ok=True)
        
        # Create the output file path
        output_txt_file_path = os.path.join(output_folder, f"{base_name}_output.mmd")

        # Save the generated output
        with open(output_txt_file_path, 'w', encoding='utf-8') as out_file:
            out_file.write(generated_text)

        print(f"🎉 Output saved to {output_txt_file_path}")

    except Exception as e:
        print(f"❌ Error generating response: {e}")

# === Your Inputs ===
json_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/gemini_2.5_pro_768/Biology/08_1002103647961841141690697632/08_1002103647961841141690697632.json"
txt_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/ocr_human/bio/08_1002103647961841141690697632.txt"
prompt = """
Role: You are a highly accurate Quality Assurance (QA) Engine for OCR systems.

Context: You will be provided with two documents for each task:

A JSON File: Contains text extracted from the original PDF by an LLM.
A TXT File: Contains text extracted by a human.
Objective: Your sole purpose is to determine which file—JSON or TXT—more accurately reflects the original content.

Specific Instructions:
From the JSON file, use the value of the "ocr_text" key.
From the TXT file, use the text between the <sol_start ...> and <sol_end> tags.
Compare both for accuracy and decide which one is more faithful to the source (assume you have access to the true content implicitly).
Report Your Verdict in the following table format:

Your entire output must be a single table that only includes rows where a discrepancy was found. The table must have the following four columns:

1. Question Number: The ID of the solution where the discrepancy occurred.
2. JSON Version: The full text of the solution from the JSON file ( where the discrepancy occurred).
3. TXT Version: The full text of the solution from the TXT file ( where the discrepancy occurred).
4. Discrepancy Analysis & Verdict: A two-part analysis:
Discrepancy Type: State the primary type of error (e.g., Spelling (Typo), Punctuation, Wording, Numerical Difference).
Verdict: State which version is correct and provide a brief justification (e.g., TXT is correct; "materials" is the more appropriate plural form.).
"""

send_files_and_prompt(txt_file_path, json_file_path, prompt)


🎉 Output saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/08_1002103647961841141690697632_output.mmd


In [11]:
import os
import csv

# Set the input directory where your .mmd files are located
input_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation'  # Change this to your folder path
output_mmd = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd'
output_csv = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv'

mmd_files = [f for f in os.listdir(input_dir) if f.endswith('.mmd')]
mmd_files.sort()  # Optional: sort files alphabetically

# Prepare lists to store extracted data
all_rows = []

# Flag to track if header has been written
header_written = False

# Reading all .mmd files and extracting table content
for filename in mmd_files:
    file_path = os.path.join(input_dir, filename)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

        for i, line in enumerate(lines):
            # Skip the header row in each .mmd file
            if i == 0 and not header_written:  # First file's header should be written
                # Write the header row only once
                all_rows.append(['Question Number', 'JSON Version', 'TXT Version', 'Discrepancy Analysis & Verdict'])
                header_written = True
                continue  # Skip the header row

            # Extract data between columns (based on "|")
            parts = [cell.strip() for cell in line.strip().strip('|').split('|')]
            if len(parts) == 4:  # Ensure that there are exactly 4 columns
                all_rows.append(parts)

# Write the extracted data to a combined .mmd file
with open(output_mmd, 'w', encoding='utf-8') as f:
    # Write table header (only once)
    f.write('| Question Number | JSON Version | TXT Version | Discrepancy Analysis & Verdict |\n')
    f.write('| :--- | :--- | :--- | :--- |\n')
    
    # Write each row to the .mmd file
    for row in all_rows:
        f.write('| ' + ' | '.join(row) + ' |\n')

# Write the extracted data to a combined CSV file
with open(output_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # Write the header row
    writer.writerows(all_rows)

print(f"Combined {len(mmd_files)} .mmd files into {output_mmd} and {output_csv} with {len(all_rows) - 1} rows (excluding header).")


Combined 8 .mmd files into /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd and /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv with 65 rows (excluding header).


In [ ]:
import os
import csv

# Set the input directory where your .mmd files are located
input_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation'  # Change this to your folder path
output_mmd = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd'
output_csv = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv'

mmd_files = [f for f in os.listdir(input_dir) if f.endswith('.mmd')]
mmd_files.sort()  # Optional: sort files alphabetically

# Prepare lists to store extracted data
all_rows = []

# Flag to track if header has been written
header_written = False

# Reading all .mmd files and extracting table content
for filename in mmd_files:
    file_path = os.path.join(input_dir, filename)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

        for i, line in enumerate(lines):
            # Skip the header row in each .mmd file
            if i == 0 and not header_written:  # First file's header should be written
                # Write the header row only once
                all_rows.append(['Question Number', 'JSON Version', 'TXT Version', 'Discrepancy Analysis & Verdict'])
                header_written = True
                continue  # Skip the header row

            # Extract data between columns (based on "|")
            parts = [cell.strip() for cell in line.strip().strip('|').split('|')]
            if len(parts) == 4:  # Ensure that there are exactly 4 columns
                all_rows.append(parts)

# Write the extracted data to a combined .mmd file
with open(output_mmd, 'w', encoding='utf-8') as f:
    # Write table header (only once)
    f.write('| Question Number | JSON Version | TXT Version | Discrepancy Analysis & Verdict |\n')
    f.write('| :--- | :--- | :--- | :--- |\n')
    
    # Write each row to the .mmd file
    for row in all_rows:
        f.write('| ' + ' | '.join(row) + ' |\n')

# Write the extracted data to a combined CSV file
with open(output_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # Write the header row
    writer.writerows(all_rows)

print(f"Combined {len(mmd_files)} .mmd files into {output_mmd} and {output_csv} with {len(all_rows) - 1} rows (excluding header).")


Combined 8 .mmd files into /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd and /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv with 65 rows (excluding header).


## break the docx into docs questions

In [2]:
import os
import csv

# Set your input directory containing the .mmd files
input_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation'  # Change this to your folder path
output_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd"           # <-- change this if you want a different output name

# Get all .mmd files in the directory
mmd_files = [f for f in os.listdir(input_dir) if f.endswith('.mmd')]
mmd_files.sort()  # Optional: sort files alphabetically

header = []
rows = []

for idx, filename in enumerate(mmd_files):
    with open(os.path.join(input_dir, filename), 'r', encoding='utf-8') as f:
        lines = f.readlines()
        # Remove empty lines
        lines = [line for line in lines if line.strip()]
        if len(lines) < 2:
            continue  # skip files that don't have at least header+separator
        if idx == 0:
            header = lines[:2]  # first two lines: header and separator
        # Add all data rows (after first two lines)
        rows.extend(lines[2:])

# Write to output file
with open(output_file, 'w', encoding='utf-8') as f:
    f.writelines(header)
    f.writelines(rows)

print(f"Merged {len(mmd_files)} files into {output_file}")

Merged 8 files into /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd


In [8]:
import csv

input_mmd = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd"   # The merged markdown table file
output_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv"           # <-- change this if you want a different output name

rows = []
header_found = False
with open(input_mmd, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Skip separator lines (those with only dashes and pipes)
        if set(line.replace('|', '').replace(':', '').replace('-', '')) == set():
            continue
        if line.startswith('|'):
            # Remove leading/trailing pipes and split
            parts = [cell.strip() for cell in line.strip('|').split('|')]
            if not header_found:
                header_found = True
                rows.append(parts[:4])  # keep header
                continue
            # Keep only the first 4 columns
            rows.append(parts[:4])

# Write to CSV
with open(output_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

# Print number of data rows (excluding header)
print(f"Converted {input_mmd} to {output_csv} (first 4 columns only)")
print(f"Number of data rows (excluding header): {len(rows) - 1}") 

Converted /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.mmd to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/pre_evaluation/all_combined.csv (first 4 columns only)
Number of data rows (excluding header): 45


In [10]:
import os
import re
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def clean_tags(content):
    # Regex to remove any tags like <sol_start>, <sol_end>, <sub_id>, etc.
    content = re.sub(r'<[^>]+>', '', content)  # Remove anything between '<' and '>'
    return content.strip()

def process_sol_start_end(doc):
    # Initialize section_id
    section_id = 1
    sections = []
    current_section = ""
    in_section = False

    # Iterate through each paragraph and apply regex for <sol_start> and <sol_end>
    for para in doc.paragraphs:
        # Process <sol_start> tags
        if '<sol_start' in para.text:
            if in_section:  # Save the previous section if any
                sections.append(clean_tags(current_section))  # Clean the tags before saving
            current_section = f"<sol_start id={section_id}>"
            section_id += 1
            in_section = True
        elif '<sol_end>' in para.text:
            current_section += " " + para.text.split('<sol_end>')[0]  # Take the part before <sol_end>
            sections.append(clean_tags(current_section))  # Save the cleaned section content
            in_section = False
        elif in_section:
            # Inside a section, add content, including <sub_id> elements
            current_section += " " + para.text.strip()

    # Handle case if the last section is not saved
    if in_section:
        sections.append(clean_tags(current_section))  # Clean the tags before saving

    return sections

def extract_and_save_sections(docx_file_path, output_folder):
    # Load the document using python-docx
    doc = Document(docx_file_path)

    # Process the document to extract sections between <sol_start> and <sol_end>
    sections = process_sol_start_end(doc)

    # Create a folder named after the file (without extension)
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    main_folder = os.path.join(output_folder, file_name)
    create_folder_if_not_exists(main_folder)

    # Save each section as a new document starting from section_1.docx
    for idx, section in enumerate(sections, start=1):
        section_doc = Document()
        section_doc.add_paragraph(section)
        section_doc.save(os.path.join(main_folder, f"section_{idx}.docx"))

    print(f"Sections saved in: {main_folder}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio"

# Run the function
extract_and_save_sections(docx_file_path, output_folder)


Sections saved in: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/06_10021024301039611141693746957


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


Overall CER: 0.9679351324735582
        CER                                     highlight_diff
0  0.032967  2[-.-][+)+]Powerisinverselyproportionaltofocal...
1  0.279476  3[-.-][+)+]Given:Objectisplaced5cminfrontofthe...
2  0.166667                              1[-.-][+)+]Convexlens
3  1.176471  4[+)+][+A+][+G+][+i+][+v+][+e+][+n+][+:+][+v+]...
4  0.244444  [+A+][+n+][+s+][+-+]2[-.-]Convexlensof[-1-][-0...


In [16]:
import os
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def process_sol_start_end(doc):
    # Initialize section_id and a list to store sections
    section_id = 1
    sections = []
    current_section = ""
    in_section = False

    # Iterate through each paragraph in the document
    for para in doc.paragraphs:
        # Print paragraph content for debugging
        print(f"Processing Paragraph: {para.text}")
        
        # Process <sol_start> tags
        if '<sol_start' in para.text:
            if in_section:  # If we are in a section, save the current section
                sections.append(current_section)  # Save the content as it is
            current_section = f"<sol_start id={section_id}>\n"  # Start a new section
            section_id += 1
            in_section = True
        elif '<sol_end>' in para.text:
            current_section += " " + para.text.split('<sol_end>')[0]  # Capture the content before <sol_end>
            sections.append(current_section)  # Save the section content as it is
            in_section = False
        elif in_section:
            # Inside a section, add content, preserving any blank lines or spaces
            current_section += para.text + "\n"

    # Handle case if the last section is not saved
    if in_section:
        sections.append(current_section)  # Save the final section as it is

    return sections

def extract_and_save_sections(docx_file_path, output_folder):
    # Load the document using python-docx
    doc = Document(docx_file_path)

    # Process the document to extract sections between <sol_start> and <sol_end>
    sections = process_sol_start_end(doc)

    # Create a folder named after the file (without extension)
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    main_folder = os.path.join(output_folder, file_name)
    create_folder_if_not_exists(main_folder)

    # Save each section as a new document starting from section_1.docx
    for idx, section in enumerate(sections, start=1):
        section_doc = Document()
        section_doc.add_paragraph(section)
        section_doc.save(os.path.join(main_folder, f"section_{idx}.docx"))

    print(f"Sections saved in: {main_folder}")


# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio"

# Run the function
extract_and_save_sections(docx_file_path, output_folder)


Processing Paragraph: <page_start no = 1>
Processing Paragraph: <sol_start id=1>
Processing Paragraph: 1.	(4) Both (1) & (2)
Processing Paragraph: <sol_end>
Processing Paragraph: 
Processing Paragraph: <sol_start id=2>
Processing Paragraph: 2.	(2) Mitochondria only
Processing Paragraph: <sol_end>
Processing Paragraph: 
Processing Paragraph: <sol_start id=3>
Processing Paragraph: 3.	(1) RBC
Processing Paragraph: <sol_end>
Processing Paragraph: 
Processing Paragraph: <sol_start id=4>
Processing Paragraph: 4.	(2) Transpiration
Processing Paragraph: <sol_end>
Processing Paragraph: 
Processing Paragraph: <sol_start id=5>
Processing Paragraph: 5.	(4) (1) & (2)
Processing Paragraph: <sol_end>
Processing Paragraph: 
Processing Paragraph: <sol_start id=6>
Processing Paragraph: 6.	(4) None of these
Processing Paragraph: <sol_end>
Processing Paragraph: <page_end>
Processing Paragraph: 
Processing Paragraph: <page_start no = 2>
Processing Paragraph: <sol_start id=7>
Processing Paragraph: 7.	
Proce

In [20]:
import os
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def process_sol_start_end(doc):
    # Initialize section_id and a list to store sections
    section_id = 1
    sections = []
    current_section = ""
    in_section = False

    # Iterate through each paragraph in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Process <sol_start> tags
        if '<sol_start' in para.text:
            if in_section:  # If we are in a section, save the current section
                sections.append(current_section)  # Save the content as it is
            current_section = f"<sol_start id={section_id}>\n"  # Start a new section
            section_id += 1
            in_section = True
        elif '<sol_end>' in para.text:
            current_section += para.text.split('<sol_end>')[0]  # Capture the content before <sol_end>
            sections.append(current_section)  # Save the section content as it is
            in_section = False
        elif in_section:
            # Inside a section, add content, preserving any blank lines or spaces
            current_section += para.text + "\n"  # Append the paragraph content, even if blank

    # Handle case if the last section is not saved
    if in_section:
        sections.append(current_section)  # Save the final section as it is

    return sections

def extract_and_save_sections(docx_file_path, output_folder):
    # Load the document using python-docx
    doc = Document(docx_file_path)

    # Process the document to extract sections between <sol_start> and <sol_end>
    sections = process_sol_start_end(doc)

    # Create a folder named after the file (without extension)
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    main_folder = os.path.join(output_folder, file_name)
    create_folder_if_not_exists(main_folder)

    # Save each section as a new document starting from section_1.docx
    for idx, section in enumerate(sections, start=1):
        section_doc = Document()
        section_doc.add_paragraph(section)
        section_doc.save(os.path.join(main_folder, f"section_{idx}.docx"))

    print(f"Sections saved in: {main_folder}")


# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio"

# Run the function
extract_and_save_sections(docx_file_path, output_folder)


Processing Paragraph: ''<page_start no = 1>''
Processing Paragraph: ''<sol_start id=1>''
Processing Paragraph: ''1.\t(4) Both (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=2>''
Processing Paragraph: ''2.\t(2) Mitochondria only''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=3>''
Processing Paragraph: ''3.\t(1) RBC''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=4>''
Processing Paragraph: ''4.\t(2) Transpiration''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=5>''
Processing Paragraph: ''5.\t(4) (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=6>''
Processing Paragraph: ''6.\t(4) None of these''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''<page_end>''
Processing Paragraph: ''''
Pr

## break the json into docs questions

In [2]:
import os
import json
from docx import Document

# Function to create a .docx file with question text
def create_docx_for_question(question_text, folder_path, section_number):
    # Create a new Document
    doc = Document()
    
    # Add the question text to the document
    doc.add_paragraph(question_text)
    
    # Define the path to save the .docx file
    docx_path = os.path.join(folder_path, f"section_{section_number}.docx")
    
    # Save the .docx file
    doc.save(docx_path)
    print(f"Created: {docx_path}")

# Function to process the JSON and create corresponding .docx files
def process_json_and_create_docx(json_file_path, output_folder):
    # Read the JSON file
    with open(json_file_path, 'r') as json_file:
        data = json.load(json_file)

    # Get the base name of the JSON file to create a folder with the same name
    base_name = os.path.splitext(os.path.basename(json_file_path))[0]

    # Create a folder for the current JSON file
    folder_path = os.path.join(output_folder, base_name)
    os.makedirs(folder_path, exist_ok=True)
    
    # Loop through the questions in the JSON
    for item in data:
        # Extract question text and number
        question_text = item.get('question_text', '')
        question_number = item.get('question_number', '')
        
        # Create a .docx for each question
        create_docx_for_question(question_text, folder_path, question_number)

# Folder paths
json_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/gemini_2.5_pro_768/Biology"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768"

# Loop through all .json files in the folder
for subfolder in os.listdir(json_folder):
    subfolder_path = os.path.join(json_folder, subfolder)
    
    # If it is a folder, loop through its contents to find .json files
    if os.path.isdir(subfolder_path):
        for filename in os.listdir(subfolder_path):
            if filename.endswith(".json"):  # Check if it's a .json file
                # Define the full path to the .json file
                json_file_path = os.path.join(subfolder_path, filename)
                
                # Process the JSON file and create corresponding .docx files
                process_json_and_create_docx(json_file_path, output_folder)


Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450/section_1.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450/section_2.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450/section_3.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450/section_4.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/08_1002103647961841141690697632/section_1.docx
Created: /

## the table code


In [7]:
import os
from docx import Document
import re

# Function to extract the question text without <sol_start id=...> and <sol_end> tags
def extract_question_text(docx_path):
    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Only keep the text between <sol_start> and <sol_end> tags (without the tags themselves)
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text)
        if match:
            question_text.append(match.group(1))  # Extract only the question text
    
    return question_text

# Function to create the comparison table in a new .docx file
def create_comparison_table(gemini_ocr_path, human_ocr_path, output_folder, section_id):
    # Extract text from both Gemini OCR and Human OCR files
    gemini_ocr_text = extract_question_text(gemini_ocr_path)
    human_ocr_text = extract_question_text(human_ocr_path)
    
    # Create a folder for the current section_id
    folder_path = os.path.join(output_folder, section_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Create a new Document for the comparison table
    doc = Document()
    doc.add_heading(f'Comparison of Section {section_id}', 0)
    
    # Create a table with 3 columns: gemini_ocr, human_ocr, CER
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Table Grid'
    
    # Set table headers
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'Gemini OCR'
    hdr_cells[1].text = 'Human OCR'
    hdr_cells[2].text = 'Character Error Rate (%)'
    
    # Fill in the table with the extracted text
    num_questions = max(len(gemini_ocr_text), len(human_ocr_text))
    for i in range(num_questions):
        row_cells = table.add_row().cells
        # Gemini OCR text
        row_cells[0].text = gemini_ocr_text[i] if i < len(gemini_ocr_text) else ''
        # Human OCR text
        row_cells[1].text = human_ocr_text[i] if i < len(human_ocr_text) else ''
        # CER column: leave empty for now
        row_cells[2].text = ''
    
    # Save the comparison table in a new .docx file
    comparison_docx_path = os.path.join(folder_path, f"comparison_{section_id}.docx")
    doc.save(comparison_docx_path)
    print(f"Created: {comparison_docx_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/01_1002115268961841141690701450"

# Folder where the comparison tables will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables"

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):  # Check if it's a .docx file
        section_id = os.path.splitext(gemini_filename)[0]  # Get the section_id from the filename
        
        # Define the full paths for Gemini OCR and Human OCR files
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        # Create the comparison table for this section
        create_comparison_table(gemini_ocr_path, human_ocr_path, output_folder, section_id)


Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables/section_1/comparison_section_1.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables/section_2/comparison_section_2.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables/section_3/comparison_section_3.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables/section_4/comparison_section_4.docx


In [13]:
import os
from docx import Document
import re

# Function to extract the question text from Human OCR (extract text between <sol_start> and <sol_end>)
# Also removes content between <sub_id> and <sub_id end> tags
def extract_question_text_human_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Only keep the text between <sol_start> and <sol_end> tags (without the tags themselves)
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text)
        if match:
            question_text_section = match.group(1)
            # Remove content between <sub_id = ...> and <sub_id end> tags but keep the text inside them
            question_text_cleaned = re.sub(r'<sub_id.*?>', '', question_text_section)  # Remove <sub_id ...>
            question_text_cleaned = re.sub(r'<sub_id end>', '', question_text_cleaned)  # Remove <sub_id end>
            question_text.append(question_text_cleaned.strip())  # Extract and clean question text
    
    return question_text

# Function to extract the full content from Gemini OCR (exactly as it appears, no changes)
def extract_question_text_gemini_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Copy the exact text in Gemini OCR (no modification)
        question_text.append(para.text)  # Add the entire paragraph text
    
    return question_text

# Function to create the comparison table in a new .mmd (Markdown) file
def create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, section_id):
    # Extract text from both Gemini OCR and Human OCR files
    gemini_ocr_text = extract_question_text_gemini_ocr(gemini_ocr_path)
    human_ocr_text = extract_question_text_human_ocr(human_ocr_path)
    
    # Create a folder for the current section_id
    folder_path = os.path.join(output_folder, section_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Create the .mmd file for the comparison table
    mmd_file_path = os.path.join(folder_path, f"comparison_{section_id}.mmd")
    
    with open(mmd_file_path, 'w') as mmd_file:
        # Add heading for the table
        mmd_file.write(f"# Comparison of Section {section_id}\n\n")
        
        # Write the table header in Markdown format
        mmd_file.write("| Human OCR | Gemini OCR | Character Error Rate (%) |\n")
        mmd_file.write("|-----------|------------|--------------------------|\n")
        
        # Ensure only one row per section (Gemini and Human OCR)
        if human_ocr_text and gemini_ocr_text:
            # Write the row for Human OCR in the first column
            mmd_file.write(f"| {human_ocr_text[0]} | {gemini_ocr_text[0]} |  | \n")
        else:
            mmd_file.write("|  |  |  | \n")  # Empty row if there's no text for one of them
    
    print(f"Created: {mmd_file_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/01_1002115268961841141690701450"

# Folder where the comparison Markdown files will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd"

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):  # Check if it's a .docx file
        section_id = os.path.splitext(gemini_filename)[0]  # Get the section_id from the filename
        
        # Define the full paths for Gemini OCR and Human OCR files
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        # Create the comparison table in Markdown for this section
        create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, section_id)


Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_1/comparison_section_1.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_2/comparison_section_2.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_3/comparison_section_3.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_4/comparison_section_4.mmd


In [15]:
import os
from docx import Document
import re

# Function to extract the question text from Human OCR (extract text between <sol_start> and <sol_end>)
# Also removes content between <sub_id> and <sub_id end> tags
def extract_question_text_human_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Only keep the text between <sol_start> and <sol_end> tags (without the tags themselves)
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text, re.DOTALL)
        if match:
            question_text_section = match.group(1)
            # Remove content between <sub_id = ...> and <sub_id end> tags but keep the text inside them
            question_text_cleaned = re.sub(r'<sub_id.*?>', '', question_text_section)  # Remove <sub_id ...>
            question_text_cleaned = re.sub(r'<sub_id end>', '', question_text_cleaned)  # Remove <sub_id end>
            # Replace multiple newlines with a space to ensure clean formatting in the table
            question_text_cleaned = re.sub(r'\n+', ' ', question_text_cleaned)
            question_text.append(question_text_cleaned.strip())  # Extract and clean question text
    
    return question_text

# Function to extract the full content from Gemini OCR (exactly as it appears, no changes)
def extract_question_text_gemini_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Copy the exact text in Gemini OCR (no modification)
        question_text.append(para.text)  # Add the entire paragraph text
    
    return question_text

# Function to create the comparison table in a new .mmd (Markdown) file
def create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, section_id):
    # Extract text from both Gemini OCR and Human OCR files
    gemini_ocr_text = extract_question_text_gemini_ocr(gemini_ocr_path)
    human_ocr_text = extract_question_text_human_ocr(human_ocr_path)
    
    # Create a folder for the current section_id
    folder_path = os.path.join(output_folder, section_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Create the .mmd file for the comparison table
    mmd_file_path = os.path.join(folder_path, f"comparison_{section_id}.mmd")
    
    with open(mmd_file_path, 'w') as mmd_file:
        # Add heading for the table
        mmd_file.write(f"# Comparison of Section {section_id}\n\n")
        
        # Write the table header in Markdown format
        mmd_file.write("| Human OCR | Gemini OCR | Character Error Rate (%) |\n")
        mmd_file.write("|-----------|------------|--------------------------|\n")
        
        # Loop through both Gemini OCR and Human OCR text and write them in the respective columns
        for human_text, gemini_text in zip(human_ocr_text, gemini_ocr_text):
            # Clean text further to make sure it's single-line and has no unwanted newlines
            human_text_cleaned = re.sub(r'\n+', ' ', human_text).strip()
            gemini_text_cleaned = re.sub(r'\n+', ' ', gemini_text).strip()
            
            mmd_file.write(f"| {human_text_cleaned} | {gemini_text_cleaned} |  | \n")
        
    print(f"Created: {mmd_file_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/01_1002115268961841141690701450"

# Folder where the comparison Markdown files will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd"

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):  # Check if it's a .docx file
        section_id = os.path.splitext(gemini_filename)[0]  # Get the section_id from the filename
        
        # Define the full paths for Gemini OCR and Human OCR files
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        # Create the comparison table in Markdown for this section
        create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, section_id)


Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_1/comparison_section_1.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_2/comparison_section_2.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_3/comparison_section_3.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/comparison_tables_mmd/section_4/comparison_section_4.mmd


In [16]:
import os
from docx import Document
import re

# Function to extract the question text from Human OCR (extract text between <sol_start> and <sol_end>)
# Also removes content between <sub_id> and <sub_id end> tags
def extract_question_text_human_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Only keep the text between <sol_start> and <sol_end> tags (without the tags themselves)
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text, re.DOTALL)
        if match:
            question_text_section = match.group(1)
            # Remove content between <sub_id = ...> and <sub_id end> tags but keep the text inside them
            question_text_cleaned = re.sub(r'<sub_id.*?>', '', question_text_section)  # Remove <sub_id ...>
            question_text_cleaned = re.sub(r'<sub_id end>', '', question_text_cleaned)  # Remove <sub_id end>
            # Replace multiple newlines with a space to ensure clean formatting in the table
            question_text_cleaned = re.sub(r'\n+', ' ', question_text_cleaned)
            question_text.append(question_text_cleaned.strip())  # Extract and clean question text
    
    return question_text

# Function to extract the full content from Gemini OCR (exactly as it appears, no changes)
def extract_question_text_gemini_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    # Open the .docx file
    doc = Document(docx_path)
    
    question_text = []
    
    # Loop through paragraphs in the document
    for para in doc.paragraphs:
        # Copy the exact text in Gemini OCR (no modification)
        question_text.append(para.text)  # Add the entire paragraph text
    
    return question_text

# Function to create the comparison table in a new .mmd (Markdown) file
def create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id):
    # Extract text from both Gemini OCR and Human OCR files
    gemini_ocr_text = extract_question_text_gemini_ocr(gemini_ocr_path)
    human_ocr_text = extract_question_text_human_ocr(human_ocr_path)
    
    # Create a folder for the current section_id (inside the subfolder named by human OCR folder)
    section_folder_path = os.path.join(output_folder, folder_name)
    os.makedirs(section_folder_path, exist_ok=True)
    
    # Create the .mmd file for the comparison table inside the subfolder
    mmd_file_path = os.path.join(section_folder_path, f"section_{section_id}_table.mmd")
    
    with open(mmd_file_path, 'w') as mmd_file:
        # Add heading for the table
        mmd_file.write(f"# Comparison of Section {section_id}\n\n")
        
        # Write the table header in Markdown format
        mmd_file.write("| Human OCR | Gemini OCR | Character Error Rate (%) |\n")
        mmd_file.write("|-----------|------------|--------------------------|\n")
        
        # Loop through both Gemini OCR and Human OCR text and write them in the respective columns
        for human_text, gemini_text in zip(human_ocr_text, gemini_ocr_text):
            # Clean text further to make sure it's single-line and has no unwanted newlines
            human_text_cleaned = re.sub(r'\n+', ' ', human_text).strip()
            gemini_text_cleaned = re.sub(r'\n+', ' ', gemini_text).strip()
            
            mmd_file.write(f"| {human_text_cleaned} | {gemini_text_cleaned} |  | \n")
        
    print(f"Created: {mmd_file_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/01_1002115268961841141690701450"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/01_1002115268961841141690701450"

# Folder where the comparison Markdown files will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Create a main folder 'tables' if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):  # Check if it's a .docx file
        section_id = os.path.splitext(gemini_filename)[0]  # Get the section_id from the filename
        
        # Define the full paths for Gemini OCR and Human OCR files
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        # Get the folder name from the human OCR folder path
        folder_name = os.path.basename(human_ocr_folder)
        
        # Create the comparison table in Markdown for this section
        create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id)


Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/01_1002115268961841141690701450/section_section_1_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/01_1002115268961841141690701450/section_section_2_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/01_1002115268961841141690701450/section_section_3_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/01_1002115268961841141690701450/section_section_4_table.mmd


In [25]:
import numpy as np

def calculate_cer(reference, predicted):
    # Length of both strings
    len_ref = len(reference)
    len_pred = len(predicted)

    # Create a 2D matrix to store the results of the comparison (Levenshtein distance)
    dp = np.zeros((len_ref + 1, len_pred + 1), dtype=int)

    # Initialize the dp matrix for base cases
    for i in range(len_ref + 1):
        dp[i][0] = i  # Deletions
    for j in range(len_pred + 1):
        dp[0][j] = j  # Insertions

    # Populate the dp matrix using the Levenshtein algorithm
    for i in range(1, len_ref + 1):
        for j in range(1, len_pred + 1):
            cost = 0 if reference[i - 1] == predicted[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1,    # Deletion
                           dp[i][j - 1] + 1,    # Insertion
                           dp[i - 1][j - 1] + cost)  # Substitution

    # The value at dp[len_ref][len_pred] gives us the total number of errors (substitutions, insertions, deletions)
    errors = dp[len_ref][len_pred]
    
    # CER is calculated as the number of errors divided by the total number of characters in the reference
    cer = errors / len_ref * 100  # Multiply by 100 for percentage
    
    return cer

# Example usage:
reference_text = "hello world"
predicted_text = "l dwddddrelede"
cer = calculate_cer(reference_text, predicted_text)
print(f"Character Error Rate (CER): {cer:.2f}%")


Character Error Rate (CER): 100.00%


In [54]:
## THIS IS TO CHANGE TO MMD FILE
import os
from docx import Document
import re
import numpy as np

def calculate_cer(reference, predicted):
    len_ref = len(reference)
    len_pred = len(predicted)

    dp = np.zeros((len_ref + 1, len_pred + 1), dtype=int)

    # Initialize the dp matrix for base cases
    for i in range(len_ref + 1):
        dp[i][0] = i  # Deletions
    for j in range(len_pred + 1):
        dp[0][j] = j  # Insertions

    # Populate the dp matrix using the Levenshtein algorithm
    for i in range(1, len_ref + 1):
        for j in range(1, len_pred + 1):
            cost = 0 if reference[i - 1] == predicted[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1,    # Deletion
                           dp[i][j - 1] + 1,    # Insertion
                           dp[i - 1][j - 1] + cost)  # Substitution

    # The value at dp[len_ref][len_pred] gives us the total number of errors
    errors = dp[len_ref][len_pred]
    
    # Cap errors at the length of the reference to ensure CER doesn't exceed 100%
    errors = min(errors, len_ref)  # Cap errors at the reference length
    
    cer = (errors / len_ref) * 100  # Multiply by 100 for percentage
    
    return cer


# Function to extract question text from Human OCR (text between <sol_start> and <sol_end>)
def extract_question_text_human_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    doc = Document(docx_path)
    question_text = []

    for para in doc.paragraphs:
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text, re.DOTALL)
        if match:
            question_text_section = match.group(1)
            question_text_cleaned = re.sub(r'<sub_id.*?>', '', question_text_section)
            question_text_cleaned = re.sub(r'<sub_id end>', '', question_text_cleaned)
            question_text_cleaned = re.sub(r'\n+', ' ', question_text_cleaned)
            question_text.append(question_text_cleaned.strip())
    
    return question_text

# Function to extract the content from Gemini OCR (exactly as it appears)
def extract_question_text_gemini_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    doc = Document(docx_path)
    question_text = []

    for para in doc.paragraphs:
        question_text.append(para.text)
    
    return question_text

# Function to create the comparison table in a .mmd (Markdown) file
def create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id):
    gemini_ocr_text = extract_question_text_gemini_ocr(gemini_ocr_path)
    human_ocr_text = extract_question_text_human_ocr(human_ocr_path)
    
    section_folder_path = os.path.join(output_folder, folder_name)
    os.makedirs(section_folder_path, exist_ok=True)
    
    # Path to the markdown file
    mmd_file_path = os.path.join(section_folder_path, f"section_{section_id}_table.mmd")
    
    with open(mmd_file_path, 'w') as mmd_file:
        mmd_file.write(f"# Comparison of Section {section_id}\n\n")
        
        # Table header
        mmd_file.write("| Human OCR | Gemini OCR | Character Error Rate (%) |\n")
        mmd_file.write("|-----------|------------|--------------------------|\n")
        
        # Loop through both Gemini OCR and Human OCR text and write them in respective columns
        # Ensure we handle cases where one text is missing
        for i in range(max(len(human_ocr_text), len(gemini_ocr_text))):
            human_text_cleaned = human_ocr_text[i] if i < len(human_ocr_text) else ""
            gemini_text_cleaned = gemini_ocr_text[i] if i < len(gemini_ocr_text) else ""
            
            human_text_cleaned = re.sub(r'\n+', ' ', human_text_cleaned).strip()
            gemini_text_cleaned = re.sub(r'\n+', ' ', gemini_text_cleaned).strip()
            
            # If one text is missing, leave that column empty and CER as N/A
            if not human_text_cleaned:
                cer = "N/A"
                mmd_file.write(f"|  | {gemini_text_cleaned} | {cer} |\n")
            elif not gemini_text_cleaned:
                cer = "N/A"
                mmd_file.write(f"| {human_text_cleaned} |  | {cer} |\n")
            else:
                cer = calculate_cer(human_text_cleaned, gemini_text_cleaned)
                mmd_file.write(f"| {human_text_cleaned} | {gemini_text_cleaned} | {cer:.2f}% |\n")
    
    print(f"Created: {mmd_file_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/04_10021039411060911141694339166"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/04_10021039411060911141694339166"
# Folder where the comparison Markdown files will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Create 'tables' folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):
        section_id = os.path.splitext(gemini_filename)[0]
        
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        folder_name = os.path.basename(human_ocr_folder)
        
        # Create the comparison table in Markdown (.mmd) for this section
        create_comparison_table_mmd(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id)




Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_14_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_6_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_18_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_7_table.mmd
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_15_table.m

In [38]:
#THIS IS TO CHANGE TO DOCX FILE
import os
from docx import Document
import re
import numpy as np

# Function to calculate Character Error Rate (CER)
def calculate_cer(reference, predicted):
    len_ref = len(reference)
    len_pred = len(predicted)

    dp = np.zeros((len_ref + 1, len_pred + 1), dtype=int)

    # Initialize the dp matrix for base cases
    for i in range(len_ref + 1):
        dp[i][0] = i  # Deletions
    for j in range(len_pred + 1):
        dp[0][j] = j  # Insertions

    # Populate the dp matrix using the Levenshtein algorithm
    for i in range(1, len_ref + 1):
        for j in range(1, len_pred + 1):
            cost = 0 if reference[i - 1] == predicted[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1,    # Deletion
                           dp[i][j - 1] + 1,    # Insertion
                           dp[i - 1][j - 1] + cost)  # Substitution

    # The value at dp[len_ref][len_pred] gives us the total number of errors
    errors = dp[len_ref][len_pred]
    
    cer = errors / len_ref * 100  # Multiply by 100 for percentage
    return cer

# Function to extract question text from Human OCR (text between <sol_start> and <sol_end>)
def extract_question_text_human_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    doc = Document(docx_path)
    question_text = []

    for para in doc.paragraphs:
        match = re.search(r'<sol_start.*?>\s*(.*?)\s*<sol_end>', para.text, re.DOTALL)
        if match:
            question_text_section = match.group(1)
            question_text_cleaned = re.sub(r'<sub_id.*?>', '', question_text_section)
            question_text_cleaned = re.sub(r'<sub_id end>', '', question_text_cleaned)
            question_text_cleaned = re.sub(r'\n+', ' ', question_text_cleaned)
            question_text.append(question_text_cleaned.strip())
    
    return question_text

# Function to extract the content from Gemini OCR (exactly as it appears)
def extract_question_text_gemini_ocr(docx_path):
    if not os.path.exists(docx_path):
        print(f"Warning: {docx_path} does not exist.")
        return []

    doc = Document(docx_path)
    question_text = []

    for para in doc.paragraphs:
        question_text.append(para.text)
    
    return question_text

# Function to create the comparison table in a .docx file
def create_comparison_table_docx(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id):
    gemini_ocr_text = extract_question_text_gemini_ocr(gemini_ocr_path)
    human_ocr_text = extract_question_text_human_ocr(human_ocr_path)
    
    section_folder_path = os.path.join(output_folder, folder_name)
    os.makedirs(section_folder_path, exist_ok=True)
    
    # Create a new Document (for .docx file)
    doc = Document()
    doc.add_heading(f"Comparison of Section {section_id}", 0)
    
    # Create table with 3 columns: Human OCR, Gemini OCR, CER
    table = doc.add_table(rows=1, cols=3)
    
    # Set the headers
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'Human OCR'
    hdr_cells[1].text = 'Gemini OCR'
    hdr_cells[2].text = 'Character Error Rate (%)'
    
    # Loop through both Gemini OCR and Human OCR text and write them in respective columns
    # Ensure we handle cases where one text is missing
    for i in range(max(len(human_ocr_text), len(gemini_ocr_text))):
        human_text_cleaned = human_ocr_text[i] if i < len(human_ocr_text) else ""
        gemini_text_cleaned = gemini_ocr_text[i] if i < len(gemini_ocr_text) else ""
        
        human_text_cleaned = re.sub(r'\n+', ' ', human_text_cleaned).strip()
        gemini_text_cleaned = re.sub(r'\n+', ' ', gemini_text_cleaned).strip()
        
        # If one text is missing, leave that column empty and CER as N/A
        if not human_text_cleaned:
            cer = "N/A"
            row_cells = table.add_row().cells
            row_cells[0].text = ""
            row_cells[1].text = gemini_text_cleaned
            row_cells[2].text = cer
        elif not gemini_text_cleaned:
            cer = "N/A"
            row_cells = table.add_row().cells
            row_cells[0].text = human_text_cleaned
            row_cells[1].text = ""
            row_cells[2].text = cer
        else:
            cer = calculate_cer(human_text_cleaned, gemini_text_cleaned)
            row_cells = table.add_row().cells
            row_cells[0].text = human_text_cleaned
            row_cells[1].text = gemini_text_cleaned
            row_cells[2].text = f"{cer:.2f}%"
    
    # Save the document
    doc_path = os.path.join(section_folder_path, f"section_{section_id}_table.docx")
    doc.save(doc_path)
    
    print(f"Created: {doc_path}")

# Paths for Gemini OCR and Human OCR directories
gemini_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/gemini_ocr_bio_768/04_10021039411060911141694339166"
human_ocr_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/human_ocr_bio/04_10021039411060911141694339166"
# Folder where the comparison Word files will be saved
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Create 'tables' folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Loop through all .docx files in the Gemini OCR folder
for gemini_filename in os.listdir(gemini_ocr_folder):
    if gemini_filename.endswith(".docx"):
        section_id = os.path.splitext(gemini_filename)[0]
        
        gemini_ocr_path = os.path.join(gemini_ocr_folder, gemini_filename)
        human_ocr_path = os.path.join(human_ocr_folder, f"{section_id}.docx")
        
        folder_name = os.path.basename(human_ocr_folder)
        
        # Create the comparison table in Word (.docx) for this section
        create_comparison_table_docx(gemini_ocr_path, human_ocr_path, output_folder, folder_name, section_id)



Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_14_table.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_6_table.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_18_table.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_7_table.docx
Created: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables/04_10021039411060911141694339166/section_section_15_tab

In [41]:
import numpy as np

def calculate_cer(reference, predicted):
    len_ref = len(reference)
    len_pred = len(predicted)

    dp = np.zeros((len_ref + 1, len_pred + 1), dtype=int)

    # Initialize the dp matrix for base cases
    for i in range(len_ref + 1):
        dp[i][0] = i  # Deletions
    for j in range(len_pred + 1):
        dp[0][j] = j  # Insertions

    # Populate the dp matrix using the Levenshtein algorithm
    for i in range(1, len_ref + 1):
        for j in range(1, len_pred + 1):
            cost = 0 if reference[i - 1] == predicted[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1,    # Deletion
                           dp[i][j - 1] + 1,    # Insertion
                           dp[i - 1][j - 1] + cost)  # Substitution

    # Total errors are found at dp[len_ref][len_pred]
    errors = dp[len_ref][len_pred]
    
    # Fix for errors exceeding reference length
    if errors > len_ref:
        errors = len_ref  # Cap the errors to the reference length
    
    # Calculate the Character Error Rate (CER)
    cer = errors / len_ref * 100  # Multiply by 100 for percentage
    
    # Ensure the CER is not greater than 100%
    cer = min(cer, 100.0)
    
    return cer

# Example usage:
reference_text = "1. (1) Tonoplast"
predicted_text = "10-09-2023 Biology Q.1 -(1) Tonoplast"
cer = calculate_cer(reference_text, predicted_text)
print(f"Character Error Rate (CER): {cer:.2f}%")


Character Error Rate (CER): 100.00%


In [44]:
import Levenshtein

def calculate_cer(reference, predicted):
    # Compute the Levenshtein distance between the two strings
    errors = Levenshtein.distance(reference, predicted)
    
    # Calculate CER as the number of errors divided by the length of the reference
    cer = (errors / len(reference)) * 100
    
    return cer

# Example usage:
reference_text = "1. (1) Tonoplast"
predicted_text = "10-09-2023 Biology Q.1 -(1) Tonoplast"
cer = calculate_cer(reference_text, predicted_text)
print(f"Character Error Rate (CER): {cer:.2f}%")


Character Error Rate (CER): 131.25%


In [50]:
import os

def count_section_ids(root_folder):
    section_count = 0
    
    # Traverse through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            # Check if the file ends with ".mmd" and follows the expected pattern
            if filename.endswith("_table.mmd") and filename.startswith("section_section_"):
                section_count += 1
    
    return section_count

# Define the root folder path
root_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Get the total count of section IDs
total_sections = count_section_ids(root_folder)
print(f"Total number of sections: {total_sections}")


Total number of sections: 69


In [51]:
import os
import re

def calculate_average_cer(root_folder):
    total_cer = 0
    file_count = 0
    
    # Traverse through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            # Check if the file ends with ".mmd"
            if filename.endswith("_table.mmd"):
                file_count += 1
                file_path = os.path.join(dirpath, filename)
                
                # Open the file and read its contents
                with open(file_path, 'r') as file:
                    content = file.read()
                    
                    # Use regex to extract CER values
                    cer_match = re.search(r'\|.*\|.*\| (\d+\.\d+)% \|', content)
                    
                    if cer_match:
                        cer_value = float(cer_match.group(1))
                        total_cer += cer_value
                    else:
                        print(f"Warning: No CER found in {filename}")
    
    # Calculate average CER
    if file_count > 0:
        average_cer = total_cer / file_count
        return average_cer
    else:
        return 0

# Define the root folder path
root_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Get the average CER for all .mmd files
average_cer = calculate_average_cer(root_folder)
print(f"Average Character Error Rate (CER) across all sections: {average_cer:.2f}%")


Average Character Error Rate (CER) across all sections: 502.08%


In [52]:
import os
import re

def calculate_average_cer_and_check_high_cer(root_folder):
    total_cer = 0
    file_count = 0
    high_cer_files = []  # List to store files with CER > 100
    
    # Traverse through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            # Check if the file ends with ".mmd"
            if filename.endswith("_table.mmd"):
                file_count += 1
                file_path = os.path.join(dirpath, filename)
                
                # Open the file and read its contents
                with open(file_path, 'r') as file:
                    content = file.read()
                    
                    # Use regex to extract CER values
                    cer_match = re.search(r'\|.*\|.*\| (\d+\.\d+)% \|', content)
                    
                    if cer_match:
                        cer_value = float(cer_match.group(1))
                        total_cer += cer_value
                        
                        # Check if CER is greater than 100 and add the file to the list
                        if cer_value > 100:
                            high_cer_files.append(filename)
                    else:
                        print(f"Warning: No CER found in {filename}")
    
    # Calculate average CER
    if file_count > 0:
        average_cer = total_cer / file_count
    else:
        average_cer = 0

    return average_cer, high_cer_files

# Define the root folder path
root_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Get the average CER and files with CER > 100
average_cer, high_cer_files = calculate_average_cer_and_check_high_cer(root_folder)

# Print results
print(f"Average Character Error Rate (CER) across all sections: {average_cer:.2f}%")

if high_cer_files:
    print("Files with CER greater than 100%:")
    for file in high_cer_files:
        print(f"- {file}")
else:
    print("No files with CER greater than 100%.")


Average Character Error Rate (CER) across all sections: 502.08%
Files with CER greater than 100%:
- section_section_1_table.mmd
- section_section_18_table.mmd
- section_section_12_table.mmd
- section_section_7_table.mmd
- section_section_8_table.mmd
- section_section_22_table.mmd


In [53]:
import os
import re

def calculate_average_cer_and_check_high_cer(root_folder):
    total_cer = 0
    file_count = 0
    high_cer_files = []  # List to store files with CER > 100
    
    # Traverse through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            # Check if the file ends with ".mmd"
            if filename.endswith("_table.mmd"):
                file_count += 1
                file_path = os.path.join(dirpath, filename)
                
                # Open the file and read its contents
                with open(file_path, 'r') as file:
                    content = file.read()
                    
                    # Use regex to extract CER values
                    cer_match = re.search(r'\|.*\|.*\| (\d+\.\d+)% \|', content)
                    
                    if cer_match:
                        cer_value = float(cer_match.group(1))
                        
                        # Cap the CER at 100 if it's greater than 100
                        if cer_value > 100:
                            cer_value = 100
                            high_cer_files.append(filename)  # Mark file with high CER (over 100)

                        total_cer += cer_value
                    else:
                        print(f"Warning: No CER found in {filename}")
    
    # Calculate average CER
    if file_count > 0:
        average_cer = total_cer / file_count
    else:
        average_cer = 0

    return average_cer, high_cer_files

# Define the root folder path
root_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/evaluation/tables"

# Get the average CER and files with CER > 100
average_cer, high_cer_files = calculate_average_cer_and_check_high_cer(root_folder)

# Print results
print(f"Average Character Error Rate (CER) across all sections: {average_cer:.2f}%")

if high_cer_files:
    print("Files with CER greater than 100% (treated as 100%):")
    for file in high_cer_files:
        print(f"- {file}")
else:
    print("No files with CER greater than 100%.")


Average Character Error Rate (CER) across all sections: 28.64%
Files with CER greater than 100% (treated as 100%):
- section_section_1_table.mmd
- section_section_18_table.mmd
- section_section_12_table.mmd
- section_section_7_table.mmd
- section_section_8_table.mmd
- section_section_22_table.mmd


In [24]:
import os
import zipfile
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                
    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder):
    """Process the content between <sol_start> and <sol_end>, and track image references"""
    section_id = 1
    sections = []
    current_section = ""
    in_section = False

    # Iterate through each paragraph in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Process <sol_start> tags
        if '<sol_start' in para.text:
            if in_section:  # If we are in a section, save the current section
                sections.append(current_section)  # Save the content as it is
            current_section = f"<sol_start id={section_id}>\n"  # Start a new section
            section_id += 1
            in_section = True
        elif '<sol_end>' in para.text:
            current_section += para.text.split('<sol_end>')[0]  # Capture the content before <sol_end>
            sections.append(current_section)  # Save the section content as it is
            in_section = False
        elif in_section:
            current_section += para.text + "\n"  # Append the paragraph content, even if blank

            # Check if the paragraph contains an image (the image filename is in XML attributes)
            if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
                img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
                img_file = os.path.join(image_folder, img_name)  # Get the image file path
                current_section += f"\n[IMAGE: {img_file}]\n"  # Placeholder for the image

    # Handle case if the last section is not saved
    if in_section:
        sections.append(current_section)  # Save the final section as it is

    return sections

def extract_and_save_sections(docx_file_path, output_folder, image_folder):
    """Extract sections and images from DOCX and save each question to a new DOCX"""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Process the document to extract sections with images
    sections = process_sol_start_end_with_images(doc, image_folder)

    # Create a folder to save sections
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    main_folder = os.path.join(output_folder, file_name)
    create_folder_if_not_exists(main_folder)

    # Save each section as a new document
    for idx, section in enumerate(sections, start=1):
        section_doc = Document()
        section_doc.add_paragraph(section)

        # Check if the section contains an image placeholder
        if '[IMAGE:' in section:
            # Placeholder logic to insert image back into the DOCX file
            # Find the image path from the placeholder text and add the image to the section
            img_path = section.split("[IMAGE: ")[1].split("]")[0]  # Extract image path
            section_doc.add_paragraph(f"Image: {img_path}")  # Add image reference as text
            section_doc.add_picture(img_path)  # Insert image into DOCX

        section_doc.save(os.path.join(main_folder, f"section_{idx}.docx"))

    print(f"Sections saved in: {main_folder}")




# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image"

# Run the function
extract_and_save_sections(docx_file_path, output_folder, image_folder)


Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image
Processing Paragraph: ''<page_start no = 1>''
Processing Paragraph: ''<sol_start id=1>''
Processing Paragraph: ''1.\t(4) Both (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=2>''
Processing Paragraph: ''2.\t(2) Mitochondria only''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=3>''
Processing Paragraph: ''3.\t(1) RBC''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=4>''
Processing Paragraph: ''4.\t(2) Transpiration''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=5>''
Processing Paragraph: ''5.\t(4) (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=6>''
Processing 

In [31]:
import os
import zipfile
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                
    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder):
    """Process the content between <sol_start> and <sol_end>, and track image references"""
    section_id = 1
    sections = []
    current_section = ""
    in_section = False
    image_references = []  # List to store image references and their positions

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Process <sol_start> tags
        if '<sol_start' in para.text:
            if in_section:  # If we are in a section, save the current section
                sections.append(current_section)  # Save the content as it is
            current_section = f"<sol_start id={section_id}>\n"  # Start a new section
            section_id += 1
            in_section = True
        elif '<sol_end>' in para.text:
            current_section += para.text.split('<sol_end>')[0]  # Capture the content before <sol_end>
            sections.append(current_section)  # Save the section content as it is
            in_section = False
        elif in_section:
            current_section += para.text + "\n"  # Append the paragraph content, even if blank

            # Check if the paragraph contains an image (the image filename is in XML attributes)
            if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
                img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
                img_file = os.path.join(image_folder, img_name)  # Get the image file path
                image_references.append(img_file)  # Store the image path
                current_section += f"\n[IMAGE: {img_file}]\n"  # Placeholder for the image

    # Handle case if the last section is not saved
    if in_section:
        sections.append(current_section)  # Save the final section as it is

    return sections, image_references

def extract_and_save_sections(docx_file_path, output_folder, image_folder):
    """Extract sections and images from DOCX and save each question to a new DOCX"""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Process the document to extract sections with images
    sections, image_references = process_sol_start_end_with_images(doc, image_folder)

    # Create a folder to save sections
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    main_folder = os.path.join(output_folder, file_name)
    create_folder_if_not_exists(main_folder)

    # Save each section as a new document
    for idx, section in enumerate(sections, start=1):
        section_doc = Document()
        section_doc.add_paragraph(section)

        # Check if the section contains an image placeholder
        if '[IMAGE:' in section:
            # Placeholder logic to insert image back into the DOCX file
            # Find the image path from the placeholder text and add the image to the section
            img_path = image_references[idx - 1]  # Get the correct image path for this section
            section_doc.add_paragraph(f"Image: {img_path}")  # Add image reference as text
            section_doc.add_picture(img_path)  # Insert image into DOCX

        section_doc.save(os.path.join(main_folder, f"section_{idx}.docx"))

    print(f"Sections saved in: {main_folder}")


# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image"


# Run the function

extract_and_save_sections(docx_file_path, output_folder, image_folder)


Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image
Processing Paragraph: ''<page_start no = 1>''
Processing Paragraph: ''<sol_start id=1>''
Processing Paragraph: ''1.\t(4) Both (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=2>''
Processing Paragraph: ''2.\t(2) Mitochondria only''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=3>''
Processing Paragraph: ''3.\t(1) RBC''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=4>''
Processing Paragraph: ''4.\t(2) Transpiration''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=5>''
Processing Paragraph: ''5.\t(4) (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=6>''
Processing 

In [35]:
import os
import zipfile
from docx import Document

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")
                
    print(f"Images extracted to {output_folder}")

def process_and_insert_placeholders(doc, image_folder):
    """Detect images and insert placeholders in the document"""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path
            para.add_run(f"\n[IMAGE: {img_file}]\n")  # Insert a placeholder for the image

    return doc, image_references

def process_docx_and_insert_placeholders(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document"""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Process the document to insert placeholders
    updated_doc, image_references = process_and_insert_placeholders(doc, image_folder)

    # Save the updated document with placeholders (without inserting images directly)
    updated_doc.save(os.path.join(output_folder, "updated_with_placeholders.docx"))
    
    print(f"Updated document saved to: {os.path.join(output_folder, 'updated_with_placeholders.docx')}")




# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image"

# Run the function to process the document
process_docx_and_insert_placeholders(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image
Processing Paragraph: ''<page_start no = 1>''
Processing Paragraph: ''<sol_start id=1>''
Processing Paragraph: ''1.\t(4) Both (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=2>''
Processing Paragraph: ''2.\t(2) Mitochondria only''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=3>''
Processing Paragraph: ''3.\t(1) RBC''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=4>''
Processing Paragraph: ''4.\t(2) Transpiration''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=5>''
Processing Paragraph: ''5.\t(4) (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<s

In [37]:
import os
import zipfile
from docx import Document

def create_folder_if_not_exists(folder_path):
    """Create folder if it doesn't already exist."""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")
                
    print(f"Images extracted to {output_folder}")

def process_and_insert_placeholders(doc, image_folder):
    """Detect images and insert placeholders in the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path
            
            # Insert a placeholder for the image in the paragraph text
            para.add_run(f"\n[IMAGE: {img_file}]\n")  # Placeholder for the image

    return doc, image_references

def process_docx_and_insert_placeholders(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Process the document to insert placeholders
    updated_doc, image_references = process_and_insert_placeholders(doc, image_folder)

    # Save the updated document with placeholders (without inserting images directly)
    updated_doc.save(os.path.join(output_folder, "updated_with_placeholders.docx"))
    
    print(f"Updated document saved to: {os.path.join(output_folder, 'updated_with_placeholders.docx')}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image"


# Run the function to process the document
process_docx_and_insert_placeholders(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image
Processing Paragraph: ''<page_start no = 1>''
Processing Paragraph: ''<sol_start id=1>''
Processing Paragraph: ''1.\t(4) Both (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=2>''
Processing Paragraph: ''2.\t(2) Mitochondria only''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=3>''
Processing Paragraph: ''3.\t(1) RBC''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=4>''
Processing Paragraph: ''4.\t(2) Transpiration''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<sol_start id=5>''
Processing Paragraph: ''5.\t(4) (1) & (2)''
Processing Paragraph: ''<sol_end>''
Processing Paragraph: ''''
Processing Paragraph: ''<s

In [42]:
import os
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")
                
    print(f"Images extracted to {output_folder}")

def process_and_insert_images(doc, image_folder):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path
            para.add_run(f"\n[IMAGE: {img_file}]\n")  # Insert a placeholder for the image

    return doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                # Extract image and save to new doc
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")
            else:
                # Add normal text or paragraph
                new_doc.add_paragraph(para.text)  # Insert paragraph text

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [43]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc):
    """Detect images and insert them into the document"""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content

        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Insert the image in the paragraph
            new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
            new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document"""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [44]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc, sol10_folder):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name (relationship ID)
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Insert the image in the new document
            new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
            new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                # Extract image and save to new doc
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [46]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content
        
        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name (relationship ID)
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Insert the image in the paragraph
            new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
            new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                # Extract image and save to new doc
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"


extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [48]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content

        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            print(f"Image found in paragraph: '{repr(para.text)}'")  # Debug: image detected in this paragraph
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name (relationship ID)
            print(f"Image relationship ID: {img_name}")  # Debug: log the image relationship ID
            
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Insert the image in the paragraph
            new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
            new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                # Extract image and save to new doc
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [51]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content

        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            print(f"Image found in paragraph: '{repr(para.text)}'")  # Debug: image detected in this paragraph
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name (relationship ID)
            print(f"Image relationship ID: {img_name}")  # Debug: log the image relationship ID
            
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Insert the image in the paragraph
            new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
            new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)
                            print(f"Inserting image at: {image_path}")  # Debug: Show the image path being inserted
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))  # Insert the image directly into the doc
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [55]:
import os
import zipfile
from docx import Document
from docx.shared import Inches

def create_folder_if_not_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def extract_images_from_docx(docx_path, output_folder):
    """Extract images from DOCX and save them in the output folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Open the DOCX file as a ZIP archive
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        # Iterate through the files in the DOCX archive
        for file in docx_zip.namelist():
            if file.startswith("word/media/"):
                # Extract the image files from the "word/media/" directory in DOCX
                img_name = os.path.basename(file)
                docx_zip.extract(file, output_folder)
                os.rename(os.path.join(output_folder, file), os.path.join(output_folder, img_name))
                print(f"Image extracted: {img_name}")

    print(f"Images extracted to {output_folder}")

def process_sol_start_end_with_images(doc, image_folder, new_doc):
    """Detect images and insert them into the document."""
    image_references = []  # List to store image references and their paths

    # Iterate through paragraphs in the document
    for para in doc.paragraphs:
        print(f"Processing Paragraph: '{repr(para.text)}'")  # Debugging to check content

        # Check if the paragraph contains an image (image is represented by a <a:blip> tag in XML)
        if 'word/media/' in para._element.xml:  # Detect if the paragraph contains an image
            print(f"Image found in paragraph: '{repr(para.text)}'")  # Debug: image detected in this paragraph
            img_name = para._element.xpath(".//a:blip/@r:embed")[0]  # Get the image name (relationship ID)
            print(f"Image relationship ID: {img_name}")  # Debug: log the image relationship ID
            
            img_file = os.path.join(image_folder, img_name)  # Get the image file path
            image_references.append(img_file)  # Store the image path

            # Check if the image exists before inserting
            if os.path.exists(img_file):
                print(f"Image found at: {img_file}")  # Debug: log the image path being inserted
                new_doc.add_paragraph(f"[IMAGE: {img_file}]")  # Placeholder for the image
                new_doc.add_picture(img_file, width=Inches(4))  # Add image directly into the doc
            else:
                print(f"[ERROR] Image not found at: {img_file}")  # Debug: image file does not exist

    return new_doc, image_references

def extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder):
    """Process the DOCX to insert image placeholders and save the document."""
    # Extract images from the original DOCX file
    extract_images_from_docx(docx_file_path, image_folder)

    # Load the document
    doc = Document(docx_file_path)

    # Create output folder for the sol10 section
    file_name = os.path.splitext(os.path.basename(docx_file_path))[0]
    sol10_folder = os.path.join(output_folder, f"{file_name}_sol10")
    create_folder_if_not_exists(sol10_folder)

    # New document to save the extracted section
    new_doc = Document()

    in_sol10 = False
    for para in doc.paragraphs:
        if "<sol_start id=10>" in para.text:  # Detect <sol_start id=10>
            in_sol10 = True
            new_doc.add_paragraph("<sol_start id=10>")
            continue
        if "<sol_end>" in para.text and in_sol10:  # Detect <sol_end>
            new_doc.add_paragraph("<sol_end>")
            in_sol10 = False
            break  # Stop after processing

        # If inside the <sol_start> and <sol_end> range, add paragraph content
        if in_sol10:
            # Add paragraph text
            new_doc.add_paragraph(para.text)  # Insert paragraph text

            # Check for images within paragraphs
            if 'word/media/' in para._element.xml:
                try:
                    # Extract image from paragraph and save it
                    for run in para.runs:
                        if 'graphic' in run._r.xml:  # Check if image is in this run
                            embed_id = run._r.xpath(".//a:blip/@r:embed")[0]
                            rId = run.part.related_parts[embed_id]
                            image_data = rId.blob
                            image_path = os.path.join(sol10_folder, "image_from_sol10.png")
                            with open(image_path, "wb") as img_file:
                                img_file.write(image_data)

                            # Debug: Verify that the image exists and is being inserted
                            print(f"Inserting image at: {image_path}")
                            new_doc.add_paragraph("[IMAGE INSERTED BELOW]")
                            new_doc.add_picture(image_path, width=Inches(4))  # Insert the image directly into the doc
                            break
                except Exception as e:
                    new_doc.add_paragraph(f"[Failed to extract image: {e}]")

    # Save the new DOCX file
    output_docx = os.path.join(sol10_folder, "section_sol10_with_images.docx")
    new_doc.save(output_docx)
    print(f"Extracted content between <sol_start id=10> and <sol_end> saved to:\n{output_docx}")

# Example usage
docx_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders.docx"
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image"
image_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images"

extract_sol_id_10_section_with_images(docx_file_path, output_folder, image_folder)


Image extracted: image1.png
Images extracted to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/images
Extracted content between <sol_start id=10> and <sol_end> saved to:
/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/updated_with_placeholders_sol10/section_sol10_with_images.docx


In [57]:
from docx import Document
from docx.shared import Inches
import os
import shutil

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to check and add images from the original document to the new document
def copy_images_from_paragraph(paragraph, new_doc):
    for run in paragraph.runs:
        if run._r.xml.find('graphic') != -1:  # Check if the run contains a graphic (image)
            try:
                img = run._r.graphic
                image_part = img.graphicData.pic.blipFill.blip
                image_data = image_part._get_or_add_blip()  # Get image data
                img_filename = f"image_{image_part.get('name')}"
                img_path = os.path.join(output_dir, img_filename)

                # Save the image data
                with open(img_path, 'wb') as img_file:
                    img_file.write(image_data)

                # Add image to the new document
                new_doc.add_paragraph(f"Image: {img_filename}")  # Add image placeholder or description
                new_doc.add_picture(img_path, width=Inches(2))  # Add image to new doc
            except Exception as e:
                print(f"Error while processing image: {e}")

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

        # Check for any images to copy to the new document
        copy_images_from_paragraph(paragraph, new_doc)

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_dir)

print(f"Extracted content has been saved to: {output_dir}")


Error while processing image: 'CT_R' object has no attribute 'graphic'
Extracted content has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [59]:
from docx import Document
from docx.shared import Inches
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/Screenshot_2025_07_15_at_3_48_21_PM.png'

# Normalize file path (removing non-standard characters)
output_image_path = output_image_path.replace(' ', '_')  # Replace non-breaking space with an underscore
output_image_path = output_image_path.replace(' ', '_')  # Replace spaces with underscores

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to add images from a file to the new document
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Add the uploaded image to the new document
add_image_to_doc(output_image_path, new_doc)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/Screenshot_2025_07_15_at_3_48_21_PM.png'

In [60]:
from docx import Document
from docx.shared import Inches
import zipfile
import os
import io

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Add the extracted images to the new document
for image_path in extracted_images:
    add_image_to_doc(image_path, new_doc)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [65]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())
        
        # Check if the paragraph contains an image and add the image if found
        if any(image_path for image_path in extracted_images):
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the location

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [66]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
image_added = False  # Flag to ensure image is added only once

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())
        
        # Add image once when the solution section starts
        if not image_added and extracted_images:
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the correct location
            image_added = True

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [68]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
image_added = False  # Flag to ensure image is added only once

for paragraph in doc.paragraphs:
    # Check if this paragraph contains the starting tag
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

        # Check if this paragraph contains an image and add the image if found
        if not image_added and extracted_images:
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the correct location
            image_added = True

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [70]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
image_added = False  # Flag to ensure image is added only once

for paragraph in doc.paragraphs:
    # Check if this paragraph contains the starting tag
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

        # Check if this paragraph contains an image and add the image if found
        if not image_added and extracted_images:
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the correct location
            image_added = True

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [72]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
image_added = False  # Flag to ensure image is added only once

for paragraph in doc.paragraphs:
    # Check if this paragraph contains the starting tag
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

        # If the paragraph contains an image, add the image at the correct location
        if not image_added and extracted_images:
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the correct location
            image_added = True

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [73]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content between <sol_start id=10> and <sol_end>
inside_sol = False
image_added = False  # Flag to ensure image is added only once

for paragraph in doc.paragraphs:
    # Check if this paragraph contains the starting tag
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        new_doc.add_paragraph(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Copy the paragraph text to new document
        new_doc.add_paragraph(paragraph.text.strip())

        # Check if this paragraph contains an image and add the image at the correct position
        if not image_added and extracted_images:
            add_image_to_doc(extracted_images[0], new_doc)  # Add image at the correct location
            image_added = True

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        new_doc.add_paragraph(paragraph.text.replace('<sol_end>', '').strip())

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with image has been saved to: {output_file_path}")


Extracted content with image has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [75]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Extracted content with images has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [76]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Extracted content with images has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [77]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text, including any text before the image
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Extracted content with images has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [81]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/07_100210065532425361141704635002.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=17>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=17>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text (even if it's part of <sub_id>)
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text (preserving everything)
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Extracted content with images has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [82]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")  # Placeholder for image
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text (even if it's part of <sub_id>)
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text (preserving everything, including text before image)
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Extracted content with images has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx


In [187]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")  # Placeholder for image
                image_added = True
                break
        
        if not image_added:
            # Add the paragraph text (even if it's part of <sub_id>)
            content_with_placeholders.append(paragraph.text.strip())

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
    else:
        # Add the paragraph text (preserving everything, including text before image)
        new_doc.add_paragraph(text)

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image1.png'

In [188]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # Check if the paragraph contains an image
        image_added = False
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add placeholder text for the image
                content_with_placeholders.append("[image]")  # Placeholder for image
                image_added = True
                print(f"Image placeholder added after: {paragraph.text.strip()}")  # Log for image
                break
        
        if not image_added:
            # Add the paragraph text (even if it's part of <sub_id>)
            content_with_placeholders.append(paragraph.text.strip())
            print(f"Text added: {paragraph.text.strip()}")  # Log for text

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())
        print(f"End of section added: {paragraph.text.strip()}")  # Log for end of section

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
        print(f"Image added at position {image_index}")  # Log for image insertion
    else:
        # Add the paragraph text (preserving everything, including text before image)
        new_doc.add_paragraph(text)
        print(f"Text inserted into new document: {text}")  # Log for text insertion

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/image1.png'

In [164]:
from docx import Document
from docx.shared import Inches
import zipfile
import os

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_10.docx'
output_image_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/extracted_image.png'

# Load the input docx file
doc = Document(input_file_path)

# Create a new document to save the extracted content
new_doc = Document()

# Function to extract and save images from the DOCX file
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        # Locate the image directory in the DOCX
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        
        extracted_images = []
        for image_file in image_files:
            # Read the image data from the DOCX zip archive
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            
            # Save the image to the output directory
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            
            extracted_images.append(image_filename)
        return extracted_images

# Function to add images to the new DOCX file at the correct place
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")  # Add image description
    new_doc.add_picture(image_path, width=Inches(2))  # Add image to new doc

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/')

# Extract content and images together, using placeholders for images
inside_sol = False
image_index = 0  # Index to track which image to add next
content_with_placeholders = []

for paragraph in doc.paragraphs:
    if '<sol_start id=10>' in paragraph.text:
        inside_sol = True
        content_with_placeholders.append(paragraph.text.replace('<sol_start id=10>', '').strip())

    if inside_sol:
        # We need to capture the text before the image first
        image_added = False
        # Check if the paragraph contains an image
        for run in paragraph.runs:
            if run._r.xml.find('graphic') != -1:  # If an image is found
                # Add the text first and then image placeholder
                content_with_placeholders.append(paragraph.text.strip())  # Add text before the image
                content_with_placeholders.append("[image]")  # Placeholder for image
                image_added = True
                print(f"Image placeholder added after: {paragraph.text.strip()}")  # Log for image
                break
        
        if not image_added:
            # Add the paragraph text (even if it's part of <sub_id>)
            content_with_placeholders.append(paragraph.text.strip())
            print(f"Text added: {paragraph.text.strip()}")  # Log for text

    if '<sol_end>' in paragraph.text and inside_sol:
        inside_sol = False
        content_with_placeholders.append(paragraph.text.replace('<sol_end>', '').strip())
        print(f"End of section added: {paragraph.text.strip()}")  # Log for end of section

# Add content to new DOCX
for text in content_with_placeholders:
    if text == "[image]" and image_index < len(extracted_images):
        # Add the corresponding image in place of the placeholder
        add_image_to_doc(extracted_images[image_index], new_doc)
        image_index += 1
        print(f"Image added at position {image_index}")  # Log for image insertion
    else:
        # Add the paragraph text (preserving everything, including text before image)
        new_doc.add_paragraph(text)
        print(f"Text inserted into new document: {text}")  # Log for text insertion

# Save the new document
new_doc.save(output_file_path)

print(f"Extracted content with images has been saved to: {output_file_path}")


Text added: <sol_start id=10>
Text added: <sub_id = 1>
Text added: 10.	(a)	Role of Hydrochloride acid in stomach :
Text added: (1) It kills the germs in the stomach.
Text added: (2) It creates an acidic medium for enzyme pepsin to actron protein activate.
Text added: <sub_id end>
Text added: <sub_id = 2>
Image placeholder added after: (b)
Text added: Function of lacteal – During the process of digestion, the lacteals absorb large molecules of fats and lipids from the small intestine.
Text added: <sub_id end>
Text added: <sol_end>
End of section added: <sol_end>
Text inserted into new document: 
Text inserted into new document: <sol_start id=10>
Text inserted into new document: <sub_id = 1>
Text inserted into new document: 10.	(a)	Role of Hydrochloride acid in stomach :
Text inserted into new document: (1) It kills the germs in the stomach.
Text inserted into new document: (2) It creates an acidic medium for enzyme pepsin to actron protein activate.
Text inserted into new document: <sub

In [2]:
from docx import Document

# Path to your DOCX file
docx_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"

# Load the document
doc = Document(docx_path)

# Find all tables
tables = doc.tables

print(f"Number of tables found: {len(tables)}")

# Print the contents of each table
for idx, table in enumerate(tables, 1):
    print(f"\nTable {idx}:")
    for row in table.rows:
        row_data = [cell.text for cell in row.cells]
        print("\t".join(row_data))

Number of tables found: 2

Table 1:
Respiration in plants	Respiration in plants	Respiration in animals	Respiration in animals
(i)	They do not breathe, they respire through their leaves.	(i)	They breathe air for respiration.
(ii)	Respiratory organs are absent. Respiration takes place through diffusion.	(ii)	Respiratory organs are present like lungs (in human), gills (in fishes) etc.

Table 2:
Light Reaction	Light Reaction	Dark Reaction	Dark Reaction
(i)	It takes place in the grana part of the chloroplast.	(i)	It takes place in the stroma part of the chloroplast.
(ii)	It is a photochemical phase.	(ii)	It is a biochemical phase.
(iii)	The end products are ATP and NADPH.	(iii)	The end product is glucose.


In [166]:
from docx import Document

def extract_between_markers(src_path, dest_path, start_marker, end_marker):
    doc = Document(src_path)
    new_doc = Document()
    copying = False
    found_start = False

    # Helper to copy a table
    def copy_table(table, new_doc):
        new_table = new_doc.add_table(rows=0, cols=len(table.columns))
        for row in table.rows:
            new_row = new_table.add_row().cells
            for idx, cell in enumerate(row.cells):
                new_row[idx].text = cell.text

    # Iterate through all elements (paragraphs and tables) in order
    for block in doc.element.body:
        if block.tag.endswith('tbl'):  # Table
            if copying:
                table = [t for t in doc.tables if t._element == block][0]
                copy_table(table, new_doc)
        elif block.tag.endswith('p'):  # Paragraph
            para = [p for p in doc.paragraphs if p._element == block][0]
            text = para.text
            if start_marker in text and not found_start:
                copying = True
                found_start = True
                # Optionally, include the marker line itself:
                # new_doc.add_paragraph(text)
                continue  # Skip the marker line itself
            if end_marker in text and copying:
                # Optionally, include the marker line itself:
                # new_doc.add_paragraph(text)
                break  # Stop after the first end marker
            if copying:
                new_doc.add_paragraph(text)

    new_doc.save(dest_path)
    print(f"Extracted content saved to {dest_path}")

# Usage
src_file ="/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx"

dest_file = "extracted_between_sol8.docx"
extract_between_markers(src_file, dest_file, "<sol_start id=8>", "<sol_end>")

Extracted content saved to extracted_between_sol8.docx


In [182]:
from docx import Document
from docx.shared import Inches
import zipfile
import os
# Helper to get all block items (paragraphs and tables) in order
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import Table
from docx.text.paragraph import Paragraph

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_8.docx'
output_image_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/'

doc = Document(input_file_path)
new_doc = Document()

def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        extracted_images = []
        for image_file in image_files:
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            extracted_images.append(image_filename)
        return extracted_images

def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")
    new_doc.add_picture(image_path, width=Inches(2))

def copy_table(table, new_doc):
    new_table = new_doc.add_table(rows=0, cols=len(table.columns))
    for row in table.rows:
        new_row = new_table.add_row().cells
        for idx, cell in enumerate(row.cells):
            new_row[idx].text = cell.text

def iter_block_items(parent):
    for child in parent.element.body.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, output_image_dir)
image_index = 0

inside_sol = False
found_start = False

for block in iter_block_items(doc):
    if isinstance(block, Paragraph):
        text = block.text
        if '<sol_start id=9>' in text and not found_start:
            inside_sol = True
            found_start = True
            continue
        if '<sol_end>' in text and inside_sol:
            inside_sol = False
            break
        if inside_sol:
            image_added = False
            for run in block.runs:
                if run._r.xml.find('graphic') != -1:
                    if image_index < len(extracted_images):
                        add_image_to_doc(extracted_images[image_index], new_doc)
                        image_index += 1
                    image_added = True
                    break
            if not image_added:
                new_doc.add_paragraph(text)
    elif isinstance(block, Table) and inside_sol:
        copy_table(block, new_doc)

new_doc.save(output_file_path)
print(f"Extracted content with images and tables has been saved to: {output_file_path}")

Extracted content with images and tables has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_8.docx


In [186]:
from docx import Document
from docx.shared import Inches
import zipfile
import os
import re

# Function to remove content between < and >
def clean_text(text):
    # This regex will remove anything between < and >, including the brackets themselves
    return re.sub(r'<.*?>', '', text)

# Helper to get all block items (paragraphs and tables) in order
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import Table
from docx.text.paragraph import Paragraph

# Path to the input and output files
input_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/OCR_gd_gem/Converting Handwriting PDF to Word File/Biology Word file/06_10021024301039611141693746957.docx'
output_file_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_8.docx'
output_image_dir = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/'

doc = Document(input_file_path)
new_doc = Document()

# Function to extract images from DOCX
def extract_images_from_docx(docx_file_path, output_image_dir):
    with zipfile.ZipFile(docx_file_path, 'r') as docx_zip:
        image_dir = 'word/media/'
        image_files = [f for f in docx_zip.namelist() if f.startswith(image_dir)]
        if not image_files:
            print("No images found in the DOCX file.")
            return []
        extracted_images = []
        for image_file in image_files:
            image_data = docx_zip.read(image_file)
            image_filename = os.path.join(output_image_dir, os.path.basename(image_file))
            with open(image_filename, 'wb') as img_file:
                img_file.write(image_data)
            extracted_images.append(image_filename)
        return extracted_images

# Function to add image to new document
def add_image_to_doc(image_path, new_doc):
    new_doc.add_paragraph(f"Image: {os.path.basename(image_path)}")
    new_doc.add_picture(image_path, width=Inches(2))

# Function to copy table to new document
def copy_table(table, new_doc):
    new_table = new_doc.add_table(rows=0, cols=len(table.columns))
    for row in table.rows:
        new_row = new_table.add_row().cells
        for idx, cell in enumerate(row.cells):
            new_row[idx].text = cell.text

# Helper function to iterate through blocks (paragraphs and tables)
def iter_block_items(parent):
    for child in parent.element.body.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

# Extract images from the original DOCX file
extracted_images = extract_images_from_docx(input_file_path, output_image_dir)
image_index = 0

inside_sol = False
found_start = False

for block in iter_block_items(doc):
    if isinstance(block, Paragraph):
        text = block.text
        # Clean the text to remove content between <>
        cleaned_text = clean_text(text)

        if '<sol_start id=10>' in text and not found_start:
            inside_sol = True
            found_start = True
            continue
        if '<sol_end>' in text and inside_sol:
            inside_sol = False
            break
        if inside_sol:
            image_added = False
            for run in block.runs:
                if run._r.xml.find('graphic') != -1:
                    if image_index < len(extracted_images):
                        add_image_to_doc(extracted_images[image_index], new_doc)
                        image_index += 1
                    image_added = True
                    break
            if not image_added:
                new_doc.add_paragraph(cleaned_text)  # Add the cleaned text here
    elif isinstance(block, Table) and inside_sol:
        copy_table(block, new_doc)

# Save the new document with extracted content
new_doc.save(output_file_path)
print(f"Extracted content with images and tables has been saved to: {output_file_path}")


Extracted content with images and tables has been saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/check_image/section_8.docx
